# 01 · Data profiling

Week 1 discovery for the DLD datasets. All logic lives in `src/dubai_property/profiling`; this notebook only calls it, so the same code runs from the CLI, tests and (later) Databricks.

**Before running:** download one year of transactions and rents into `data/raw/` (see `docs/data_sources.md`). Until then, the notebook falls back to the synthetic fixture.

In [ ]:
from pathlib import Path

from dubai_property.profiling import profile_dataframe, read_raw_csv, render_markdown

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Point these at your real downloads; None = skip that dataset
DATASETS = {
    "transactions": {"path": None, "key": "transaction_id"},
    "rents": {"path": None, "key": "contract_id"},
}

FALLBACK = REPO / "tests/fixtures/sample_transactions.csv"
SAMPLE_ROWS = None  # e.g. 200_000 for a quick first look at a large file

In [ ]:
profiles = {}
for name, cfg in DATASETS.items():
    path = Path(cfg["path"]) if cfg["path"] else None
    if path is None:
        if name != "transactions":
            continue
        path, name = FALLBACK, "sample_transactions (synthetic)"
    df = read_raw_csv(path, nrows=SAMPLE_ROWS)
    profiles[name] = profile_dataframe(df, name=name, key_column=cfg["key"], source_file=path.name)
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns")

In [ ]:
from IPython.display import Markdown, display

for p in profiles.values():
    display(Markdown(render_markdown(p)))

## Save reports

Commit the Markdown reports under `docs/profiles/` and copy the flags into `docs/data_dictionary.md`.

In [ ]:
out_dir = REPO / "docs/profiles"
out_dir.mkdir(parents=True, exist_ok=True)
for name, p in profiles.items():
    slug = name.split(" ")[0]
    (out_dir / f"{slug}.md").write_text(render_markdown(p), encoding="utf-8")
    print("wrote", out_dir / f"{slug}.md")